# IVtrace v1.0a-jupyter #

Программное обеспечение IVtrace предназначено для автоматизированного снятия амплитудной характеристики датчиков тока. Реализовано на языке Python в среде Jupyter Notebook. Использует библиотеки pyvisa для взаимодействия с измерительными приборами по стандарту SCPI через интерфейс NI‑VISA, pandas для накопления и экспорта данных, matplotlib для визуализации. ПО поддерживает:

- автоматическое обнаружение вольтметра (АКИП‑2101 / Siglent) и источника тока (ITECH IT‑M3910D);

- кэширование параметров измерения в JSON‑конфигурации;

- импульсный режим работы источника (включение/выключение выхода на каждой токовой точке);

- ручное переключение полярности с логическим учётом знака в результатах;

- сохранение метаданных и измеренных значений в формат CSV.

Для работы предварительно требуется установить NI-VISA (http://www.ni.com/download/ni-visa-17.5/7220/en/), а также, очевидно, python и pip

### Установка библиотек ###

In [ ]:
%pip install pyvisa pandas matplotlib

### Импорт и проверка библиотек ###

In [ ]:
import os
import re
import sys
import time
import json
from datetime import datetime
from pathlib import Path
import numpy as np
import scipy
import pyvisa
import pandas as pd
import matplotlib.pyplot as plt

print("PyVISA:", pyvisa.__version__)
print("Pandas:", pd.__version__)
print("Все библиотеки готовы.")

### Автоматическое определение устройств ###

1. Создаётся менеджер ресурсов pyvisa.ResourceManager() (без аргумента '@py' — используется системная NI‑VISA).
2. Получается список всех доступных VISA‑ресурсов. Если список пуст — работа скрипта аварийно завершается.
3. Для каждого ресурса:
- открывается сессия, задаётся кодировка utf-8 и таймаут 3 с;
- отправляется команда *IDN?, ответ анализируется на наличие ключевых слов (AKIP-2101 / SIGLENT → вольтметр; ITECH / IT-M → источник тока);
- сессия закрывается.
4. Если оба адреса не найдены — аварийное завершение.
5. Найденные адреса dmm_addr и curr_src_addr сохраняются и выводятся пользователю.

In [ ]:
rm = pyvisa.ResourceManager()
resources = rm.list_resources()

if len(resources) == 0:
    raise SystemExit("Не найдено ни одного VISA-ресурса. Проверьте подключение и драйверы.")

dmm_addr = None
curr_src_addr = None

for res in resources:
    try:
        instr = rm.open_resource(res)
        instr.encoding = 'utf-8'
        instr.timeout = 3000
        idn = instr.query('*IDN?').strip()
        print(f'{res}  ->  {idn}')
        if 'AKIP-2101' in idn or 'SIGLENT' in idn.upper():
            dmm_addr = res
        elif 'ITECH' in idn.upper() or 'IT-M' in idn.upper():
            curr_src_addr = res
        instr.close()
    except Exception as e:
        print(f'{res}  ->  Ошибка при опросе: {e}')

if not dmm_addr or not curr_src_addr:
    raise SystemExit("Не удалось обнаружить один или оба прибора. Проверьте список ресурсов выше.")

print(f"\nВольтметр: {dmm_addr}")
print(f"Источник тока: {curr_src_addr}")

### Инициализация ###

1. Определяется корневая директория сохранения `C:/IVTraceData`, при необходимости создаётся. В ней же хранится файл `ivtrace_config.json` для кэширования настроек.
2. Реализованы функции `load_config()` и `save_config()` для чтения/записи параметров в JSON.
3. При запуске:
   - загружается предыдущий конфиг (если есть);
   - пользователю отображаются сохранённые параметры: диапазон тока, шаг, ограничение напряжения, задержки на установку и охлаждение, последняя ветвь и комментарий;
   - предлагается использовать их (ввод `y/n`);
   - при отказе или отсутствии конфига последовательно запрашиваются: начальный ток, конечный ток, шаг, ограничение напряжения (`V_limit`), задержка на установку тока (`delay`), задержка на охлаждение между точками (`cooling_delay`);
   - ввод каждого числового параметра выполняется в цикле с обработкой ошибок преобразования типов.
4. Комментарий (датчик, пометка) запрашивается всегда, но с подсказкой последнего значения из конфига; можно оставить пустым или нажать Enter, чтобы сохранить прежний.
5. Выбор ветви (positive/negative) реализован отдельным диалогом с поддержкой коротких синонимов (`p`, `n`, `+`, `-`) и подсказкой последнего использованного значения. Если нажать Enter, принимается подсказанное значение.
6. Полученные параметры (включая комментарий и ветвь) немедленно сохраняются в конфигурационный файл.
7. Формируется имя CSV‑файла, включающее безопасную версию комментария (только латиница, цифры, знаки `_` и `-`), временную метку и направление, например `IVtrace_LEM_positive_20250525_143025.csv`.

In [ ]:
# ============================
# Константы и функции конфигурации
# ============================
SAVE_DIR = Path("C:/IVTraceData")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_FILE = SAVE_DIR / "ivtrace_config.json"

def load_config():
    if CONFIG_FILE.exists():
        try:
            with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"Ошибка чтения конфига: {e}")
    return None

def save_config(config):
    try:
        with open(CONFIG_FILE, 'w', encoding='utf-8') as f:
            json.dump(config, f, indent=4)
    except Exception as e:
        print(f"Ошибка сохранения конфига: {e}")

# ============================
# Ввод параметров (комментарий и ветвь всегда запрашиваются)
# ============================
print("\n=== Настройка измерения ===")

saved_config = load_config()
if saved_config:
    print("\nНайдены сохранённые параметры:")
    print(f"  Ток: {saved_config['I_start']} → {saved_config['I_stop']} А, шаг {saved_config['I_step']} А")
    print(f"  Ограничение напряжения: {saved_config['V_limit']} В")
    print(f"  Задержка на установку: {saved_config['delay']} с")
    print(f"  Задержка на охлаждение: {saved_config['cooling_delay']} с")
    print(f"  Последняя ветвь: {saved_config.get('direction', '?')}")
    print(f"  Последний комментарий: {saved_config.get('label', '')}")
    use_prev = input("\nИспользовать эти параметры? (y/n, по умолчанию y): ").strip().lower()
    if use_prev != 'n':
        I_start = saved_config['I_start']
        I_stop  = saved_config['I_stop']
        I_step  = saved_config['I_step']
        V_limit = saved_config['V_limit']
        delay   = saved_config['delay']
        cooling_delay = saved_config['cooling_delay']
        print("\nПараметры загружены.")
    else:
        I_start = I_stop = I_step = V_limit = delay = cooling_delay = None
else:
    I_start = I_stop = I_step = V_limit = delay = cooling_delay = None

if I_start is None:
    while True:
        try:
            I_start = float(input("Начальный ток (А): "))
            I_stop  = float(input("Конечный ток (А): "))
            I_step  = float(input("Шаг по току (А): "))
            V_limit = float(input("Ограничение напряжения на источнике (В): "))
            delay   = float(input("Задержка на установку тока (с): "))
            cooling_delay = float(input("Задержка на охлаждение между точками (с): "))
            break
        except ValueError as e:
            print(f"Ошибка ввода: {e}. Попробуйте снова.\n")
    # сохраним основные параметры без комментария и ветви пока
    save_config({
        'I_start': I_start,
        'I_stop': I_stop,
        'I_step': I_step,
        'V_limit': V_limit,
        'delay': delay,
        'cooling_delay': cooling_delay,
        'direction': saved_config.get('direction', '') if saved_config else '',
        'label': saved_config.get('label', '') if saved_config else ''
    })

# --- Ввод комментария (датчика) ---
last_label = saved_config.get('label', '') if saved_config else ''
if last_label:
    hint_label = f" (Enter для '{last_label}', или введите новый)"
else:
    hint_label = ""
label = input(f"Комментарий (датчик, пометка){hint_label}: ").strip()
if label == '' and last_label:
    label = last_label

# --- Ввод ветви ---
last_dir = saved_config.get('direction', '') if saved_config else ''
if last_dir:
    hint_dir = f" (Enter для {last_dir}, или введите p/n/+/-)"
else:
    hint_dir = ""
while True:
    dir_input = input(f"Ветвь (positive/p/+ или negative/n/-){hint_dir}: ").strip().lower()
    if dir_input == '' and last_dir:
        dir_input = last_dir
    if dir_input in ('positive', 'p', '+'):
        direction = 'positive'
        break
    elif dir_input in ('negative', 'n', '-'):
        direction = 'negative'
        break
    else:
        print("Некорректная ветвь. Используйте positive/p/+ или negative/n/-")

# Сохраняем полный конфиг с обновлёнными комментарием и ветвью
save_config({
    'I_start': I_start,
    'I_stop': I_stop,
    'I_step': I_step,
    'V_limit': V_limit,
    'delay': delay,
    'cooling_delay': cooling_delay,
    'label': label,
    'direction': direction
})

# Готовим безопасное имя файла
label_safe = re.sub(r'[^a-zA-Z0-9_\- ]', '', label).replace(' ', '_') if label else 'nolabel'
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = SAVE_DIR / f"IVtrace_{label_safe}_{direction}_{timestamp_str}.csv"

print(f"\nФайл результатов: {csv_filename}")
print(f"Диапазон: {I_start}..{I_stop} А, шаг {I_step} А, ограничение V={V_limit} В, ветвь: {direction}")
print(f"Комментарий: {label}")
print(f"Задержка установки: {delay} с, задержка охлаждения: {cooling_delay} с")

# ---------- 3. ПОДКЛЮЧЕНИЕ И ИНИЦИАЛИЗАЦИЯ ----------
dmm = rm.open_resource(dmm_addr)
dmm.encoding = 'utf-8'
dmm.timeout = 5000

curr = rm.open_resource(curr_src_addr)
curr.encoding = 'utf-8'
curr.timeout = 5000

curr.write('*RST')
time.sleep(1)

# === Ключевые команды ===
curr.write('SOUR:CC:PRI')          # приоритет CC (режим источника тока)
curr.write(f'SOUR:VOLT {V_limit}') # напряжение, под которое будет подниматься источник, чтобы выдать заданный ток
curr.write(f'SOUR:CURR 0')         # начальный ток — 0
curr.write('SOUR:CURR:SLEW 10')    # скорость нарастания тока 10 А/с (опционально, можно убрать)
# =========================

### Измерения ###

1. Выполняется сброс источника (`*RST`), пауза 1 с.
2. Источник переводится в режим приоритета стабилизации тока (`SOUR:CC:PRI`).
3. Устанавливается ограничение напряжения (`SOUR:VOLT {V_limit}`) и нулевой начальный ток (`SOUR:CURR 0`). Задаётся скорость нарастания тока 10 А/с (`SOUR:CURR:SLEW 10`). Выход остаётся выключенным.
4. Вольтметр настраивается на измерение постоянного тока: режим `CURR:DC`, фиксированный диапазон 10 А, время интегрирования 1 PLC.
5. Определяются знак `sign` (+1 для positive, –1 для negative) и количество шагов `num_steps = int((I_stop - I_start)/I_step) + 1`.
6. Для каждого шага `step` от 0 до `num_steps-1`:
   - вычисляется абсолютный ток `abs_current = I_start + step * I_step` (всегда ≥0);
   - знаковый ток `signed_current = abs_current * sign` (только для записи в результаты);
   - на источник подаётся команда `SOUR:CURR {abs_current}`;
   - включается выход (`OUTP ON`);
   - выдерживается задержка `delay` с на установление тока;
   - производится трёхкратное измерение тока вольтметром (`MEAS:CURR:DC?`), результаты усредняются в `i_avg`;
   - выход выключается (`OUTP OFF`);
   - выдерживается задержка `cooling_delay` с на охлаждение;
   - в список `results` добавляется запись: метка времени, `I_set_A = signed_current`, `I_meas_A = i_avg`.
7. После цикла источник устанавливает ток 0 и выключает выход (однократно).
8. Из `results` формируется `pandas.DataFrame`.
9. Файл CSV открывается с кодировкой UTF-8. В начало построчно записываются метаданные (каждая строка начинается с `#`): датчик (комментарий), диапазон тока, шаг, ограничение напряжения, ветвь, задержка установки, задержка охлаждения, время измерения, количество точек. Затем добавляется пустая строка-разделитель `#`.
10. В CSV дописываются данные из `DataFrame` (столбцы `Timestamp`, `I_set_A`, `I_meas_A`).
11. Пользователю выводятся имя файла и первые 10 строк данных.

In [ ]:
# ============================
# Измерительный цикл (импульсный режим) – измерение ТОКА с динамическим диапазоном
# ============================

print("Подключаюсь к приборам...\n")

# ---- Инициализация источника (без изменений) ----
curr.write('*RST')
time.sleep(1)
curr.write(f'SOUR:VOLT {V_limit}')
curr.write('SOUR:CURR 0')
print("Источник тока готов.\n")

# ---- Настройка вольтметра: начальный диапазон 10 А ----
dmm.write('*RST')
time.sleep(0.5)
dmm.write('SENS:FUNC "CURR:DC"')
dmm.write('SENS:CURR:DC:NPLC 1')
dmm.write('SENS:CURR:DC:RANG 10')   # начинаем с максимального диапазона
current_range = 10.0                # текущий предел в А
available_ranges = [0.0002, 0.002, 0.02, 0.2, 2.0, 10.0]  # все доступные пределы
range_idx = len(available_ranges) - 1   # индекс текущего предела (10 А)

print("Вольтметр: начальный диапазон 10 А, ожидается уточнение после первого измерения.\n")

# ---- Измерительный цикл ----
results = []
sign = -1 if direction == 'negative' else 1
num_steps = int((I_stop - I_start) / I_step) + 1

for step in range(num_steps):
    abs_current = I_start + step * I_step
    signed_current = abs_current * sign

    # Установка тока и включение
    curr.write(f'SOUR:CURR {abs_current}')
    curr.write('OUTP ON')
    time.sleep(delay)

    # Измерение (3 отсчёта)
    currents = []
    for _ in range(3):
        try:
            i = float(dmm.query('MEAS:CURR:DC?'))
            currents.append(i)
        except pyvisa.errors.VisaIOError as e:
            # Если перегрузка (overload) – значит ток превышает текущий предел
            print(f"  ⚠ Перегрузка при измерении, переключаем диапазон вверх.")
            # Увеличиваем диапазон, если возможно
            if range_idx < len(available_ranges) - 1:
                range_idx += 1
                current_range = available_ranges[range_idx]
                dmm.write(f'SENS:CURR:DC:RANG {current_range}')
                print(f"  ↓ Новый диапазон: {current_range} A")
                # Повторяем попытку измерения
                try:
                    i = float(dmm.query('MEAS:CURR:DC?'))
                    currents.append(i)
                except:
                    continue
        except Exception as e:
            print(f"  Ошибка измерения: {e}")
    i_avg = sum(currents) / len(currents) if currents else 0.0

    # ---- Динамическое переключение диапазона (после первого измерения) ----
    if step == 0:
        # Первое измерение – выбираем оптимальный предел под измеренный ток
        best_range = None
        for r in available_ranges:
            if r >= i_avg:
                best_range = r
                break
        if best_range is None:
            best_range = 10.0
        if best_range != current_range:
            current_range = best_range
            range_idx = available_ranges.index(current_range)
            dmm.write(f'SENS:CURR:DC:RANG {current_range}')
            print(f"  → После первого измерения диапазон переключён на {current_range} A")
    else:
        # Проверка выхода за пределы текущего диапазона (вверх)
        if i_avg > current_range * 0.95:  # запас 5% до перегрузки
            # Увеличиваем до следующего предела
            if range_idx < len(available_ranges) - 1:
                range_idx += 1
                new_range = available_ranges[range_idx]
                dmm.write(f'SENS:CURR:DC:RANG {new_range}')
                print(f"  ↑ Диапазон увеличен до {new_range} A (измерено {i_avg:.4f} A)")
                current_range = new_range
        # Переключение вниз (опционально) – если ток упал ниже 10% предела и есть меньший предел
        elif i_avg < current_range * 0.1 and range_idx > 0:
            # Можно уменьшить до предела, который >= i_avg
            for r in reversed(available_ranges[:range_idx]):
                if r >= i_avg:
                    range_idx = available_ranges.index(r)
                    new_range = r
                    dmm.write(f'SENS:CURR:DC:RANG {new_range}')
                    print(f"  ↓ Диапазон уменьшен до {new_range} A (измерено {i_avg:.4f} A)")
                    current_range = new_range
                    break

    # Выключение выхода и охлаждение
    curr.write('OUTP OFF')
    time.sleep(cooling_delay)

    results.append({
        'Timestamp': datetime.now().isoformat(),
        'I_set_A': signed_current,
        'I_meas_A': i_avg
    })
    print(f"  I_уст = {signed_current:+.4f} А  ->  I_изм = {i_avg:.6f} А")

# Финальное выключение
curr.write('SOUR:CURR 0')
curr.write('OUTP OFF')
print("\nИзмерения завершены, источник выключен.")

# ============================
# Сохранение в CSV (метаданные + данные)
# ============================

df = pd.DataFrame(results)

with open(csv_filename, 'w', encoding='utf-8') as f:
    f.write(f"# Датчик: {label}\n")
    f.write(f"# Диапазон заданного тока: {I_start}..{I_stop} А, шаг {I_step} А, ограничение V={V_limit} В, ветвь: {direction}\n")
    f.write(f"# Задержка установки: {delay} с\n")
    f.write(f"# Задержка охлаждения: {cooling_delay} с\n")
    f.write(f"# Время измерения: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"# Всего точек: {len(df)}\n")
    f.write("#\n")
    df.to_csv(f, index=False)

print(f"Данные сохранены в {csv_filename}")
df.head(10)

### Визуализация ###

In [ ]:
# ============================
# Построение графика и расчёт погрешности
# ============================

try:
    from scipy.interpolate import CubicSpline
    SCIPY = True
except ImportError:
    SCIPY = False
    print("⚠ scipy не установлен, сплайн будет заменён ломаной. Установите: pip install scipy")

# ---------- 1. Поиск последнего CSV-файла ----------
DATA_DIR = Path("C:/IVTraceData")
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Папка {DATA_DIR} не найдена. Сначала выполните измерения.")

csv_files = list(DATA_DIR.glob("IVtrace_*.csv"))
if not csv_files:
    raise FileNotFoundError("Нет CSV-файлов в папке IVTraceData.")
latest_file = max(csv_files, key=lambda f: f.stat().st_mtime)
print(f"Найден файл: {latest_file.name}")

# ---------- 2. Чтение данных и метаданных ----------
with open(latest_file, 'r', encoding='utf-8') as f:
    lines = f.readlines()

metadata = {}
for line in lines:
    if line.startswith('#'):
        match = re.match(r'#\s*(.*?)\s*:\s*(.*)', line)
        if match:
            key, value = match.groups()
            metadata[key.strip()] = value.strip()

df = pd.read_csv(latest_file, comment='#')
print(f"Загружено {len(df)} точек.")

label = metadata.get('Датчик', 'Неизвестный датчик')
direction = metadata.get('ветвь', '?')
V_limit = metadata.get('ограничение V', '?')
I_start = df['I_set_A'].min()
I_stop = df['I_set_A'].max()

# ---------- 3. Ввод параметров ----------
while True:
    try:
        I_nom = float(input("Номинальный первичный ток датчика (А, напр. 150): "))
        if I_nom <= 0:
            print("Ток должен быть положительным.")
            continue
        break
    except ValueError:
        print("Введите число.")

while True:
    try:
        X = float(input("Коэффициент преобразования 1:X, введите X (напр. 1500): "))
        if X <= 0:
            print("X должен быть положительным.")
            continue
        break
    except ValueError:
        print("Введите число.")

K = 1.0 / X                     # коэффициент передачи I_out / I_in
I_sec_nom = I_nom * K           # номинальный выходной ток при I_nom

# ---------- 4. Расчёт погрешности ----------
df['I_expected_A'] = df['I_set_A'] * K
df['Error_percent'] = np.abs(df['I_meas_A'] - df['I_expected_A']) / I_sec_nom * 100

# ---------- 5. Построение графиков ----------
# Настройка внешнего вида: белый фон, минорные деления
plt.style.use('default')                     # чистый стиль без серого фона
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10),
                               sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})  # верхний в 3 раза выше
fig.patch.set_facecolor('white')
ax1.set_facecolor('white')
ax2.set_facecolor('white')

# Минорные деления для частой сетки
ax1.minorticks_on()
ax2.minorticks_on()

# Верхний график: выходной ток
ax1.plot(df['I_set_A'], df['I_meas_A'], 'o-', color='steelblue', markersize=4,
         label=f'{label} ({direction}) – измер.')
ax1.plot(df['I_set_A'], df['I_expected_A'], '--', color='orange', linewidth=1.5,
         label=f'Ожидаемый (1:{int(X) if X.is_integer() else X:.1f})')
ax1.set_ylabel('Выходной ток датчика, А')
ax1.set_title(f'Амплитудная характеристика датчика тока\n'
              f'Диапазон {I_start}..{I_stop} А, $V_{{огр}}$={V_limit} В')
ax1.legend(loc='upper left')
ax1.grid(True, which='major', linestyle='-', linewidth=0.6, alpha=0.7)
ax1.grid(True, which='minor', linestyle=':', linewidth=0.4, alpha=0.5)

# Нижний график: приведённая погрешность (со сплайном, если scipy доступен)
# Сортируем данные по X для корректной интерполяции
x = df['I_set_A'].values
y = df['Error_percent'].values
order = np.argsort(x)
x_sorted = x[order]
y_sorted = y[order]

if SCIPY and len(x_sorted) > 3:
    # Строим сплайн (кубический)
    cs = CubicSpline(x_sorted, y_sorted)
    x_smooth = np.linspace(x_sorted[0], x_sorted[-1], 500)
    y_smooth = cs(x_smooth)
    ax2.plot(x_smooth, y_smooth, '-', color='firebrick', linewidth=1.2,
             label='Погрешность приведённая')
else:
    # Соединяем прямыми, если scipy нет
    ax2.plot(x_sorted, y_sorted, '-', color='firebrick', linewidth=1.2,
             label='Погрешность приведённая')

# Точки измерений
ax2.plot(df['I_set_A'], df['Error_percent'], 'x', color='firebrick', markersize=6,
         alpha=0.7)
ax2.axhline(y=0, color='gray', linewidth=0.5)
ax2.set_xlabel('Заданный первичный ток $I_{уст}$, А')
ax2.set_ylabel('Погрешность, %')
ax2.legend(loc='upper right')
ax2.grid(True, which='major', linestyle='-', linewidth=0.6, alpha=0.7)
ax2.grid(True, which='minor', linestyle=':', linewidth=0.4, alpha=0.5)

plt.tight_layout()

# ---------- Сохранение графика ----------
png_path = latest_file.with_suffix('.png')   # имя как у CSV, но .png
plt.savefig(png_path, dpi=150, bbox_inches='tight')
print(f"График сохранён: {png_path}")

plt.show()

# ---------- 6. Статистика ----------
max_err = df['Error_percent'].max()
mean_err = df['Error_percent'].mean()
print(f"\nМаксимальная приведённая погрешность: {max_err:.4f} %")
print(f"Средняя приведённая погрешность:   {mean_err:.4f} %")